### WUSTL/Olin - DAT5567 - Fall 2025 - Farahat

## Module 6 Lab Problem Set

### Template File

Amr Farahat

2025-12-04

---

This is our final lab problem set for the course and it consists of a single problem / task: to implement an optimization model that solves any 9x9 Sudoku puzzle! 

For those unfamiliar with Sudoku, here's a brief primer: 

*(Thanks, Google Gemini 3 Pro)*

**What is Sudoku?**

Sudoku is a logic-based puzzle played on a 9x9 grid. The large grid is subdivided into nine smaller 3x3 boxes (often bordered with thicker lines).

At the start of the game, some of the squares are already filled with numbers. These are your "givens" or clues. Your goal is to fill in the remaining empty squares so that the entire grid is complete.

**The 3 Golden Rules**

To solve the puzzle, you must fill every empty square with a number from 1 to 9 following these three strict constraints:

1.   **The Row Rule**: Every horizontal row must contain the numbers 1–9 exactly once. (No repeats).

2.   **The Column Rule**: Every vertical column must contain the numbers 1–9 exactly once. (No repeats).

3.   **The Box Rule**: Every marked 3x3 box must contain the numbers 1–9 exactly once. (No repeats).

If you want to learn more about the game, here's the [wikipedia article](https://en.wikipedia.org/wiki/Sudoku).

I've obtained three puzzles from today's (Deceber 4, 2025) [New York Times Sudoku page](https://www.nytimes.com/puzzles/sudoku). One is labelled "easy", the other is labelled "medium", and the third is labelled "hard". They are all located in a spreadsheet labelled sudoku_input.xlsx on Canvas. 

Your task is to formulate and implement an **integer** **linear** optimization model to solve the problem. Your code should read in the input from the excel file, solve the problem, and then display the solution as output at the bottom of this notebook. Submit an html or pdf of the fully implemented notebook with the solution for the "hard" problem showing. 

In terms of the formulation, here the only hint I'll provide you: The simplest choice of **decision variables** to implement this problem is the following:

$x_{r,c,d}$: a binary variable equal to $1$ if the square in row $r$ and column $c$ is assigned digit $d$; $0$ otherwise.


I've also set up some data structures that help you get started and ensure uniform notation across the class. Complete the code below. Good luck!

*Optional challenge questions for thought:*
(You don't need to provide answers to these questions, but I'l lask the TAs to forward me answers from students that they feel have been well-thought through.)

i.    How can you check if an alternative solution exists? Good Sudoku problems are supposed to have a unique solution. How can you verify that>

ii.   Can you think of other games / puzzles that can be solved using optimiztion modelling?

iii.  Can you use an optimization model to design new sudoku puzzles of various degrees of difficulties?

---

## Optimization using PuLP

### Step 1: Setup

#### Import required packages

In [1]:
import pandas as pd
import numpy as np
import pulp

#### Define or read-in problem parameters and data

In [2]:
input_file = "sudoku_input.xlsx"

In [3]:
df_input = pd.read_excel(input_file, sheet_name="easy", skiprows=0, nrows=9, usecols="A:I", header=None)
grid_input = df_input.fillna(0).to_numpy()

In [4]:
grid_input

array([[0., 0., 1., 6., 2., 4., 0., 0., 7.],
       [6., 9., 0., 0., 0., 7., 0., 1., 3.],
       [5., 0., 0., 0., 1., 0., 0., 8., 0.],
       [0., 6., 9., 0., 0., 1., 8., 0., 0.],
       [0., 4., 0., 0., 6., 0., 7., 9., 0.],
       [0., 5., 0., 0., 8., 2., 4., 0., 0.],
       [0., 1., 0., 7., 0., 0., 0., 2., 6.],
       [8., 0., 6., 2., 0., 0., 0., 7., 9.],
       [0., 0., 2., 1., 3., 0., 0., 0., 8.]])

In [5]:
R = [1, 2, 3, 4, 5, 6, 7, 8, 9]
C = [1, 2, 3, 4, 5, 6, 7, 8, 9]
D = [1, 2, 3, 4, 5, 6, 7, 8, 9]

In [6]:
Boxes = {"TL": {"rows": [1,2,3], "cols": [1,2,3]}, 
         "TR": {"rows": [1,2,3], "cols": [4,5,6]}, 
         "TM": {"rows": [1,2,3], "cols": [7,8,9]},
         "ML": {"rows": [4,5,6], "cols": [1,2,3]}, 
         "MM": {"rows": [4,5,6], "cols": [4,5,6]}, 
         "MR": {"rows": [4,5,6], "cols": [7,8,9]},
         "BL": {"rows": [7,8,9], "cols": [1,2,3]}, 
         "BM": {"rows": [7,8,9], "cols": [4,5,6]}, 
         "BR": {"rows": [7,8,9], "cols": [7,8,9]} 
         }

### Step 2: Create Model Object

In [27]:
model = pulp.LpProblem('Soduku', pulp.LpMinimize)

### Step 3: Add Decision Variables

In [28]:
x = pulp.LpVariable.dicts("x", (R,C,D), cat='Binary')


### Step 4: Add Objective Function and Constraints

#### Objective Function

In [29]:
model += 0, "Objective"

#### Constraints

In [30]:
# Each cell must contain exactly one value
for r in R:
    for c in C:
        model += pulp.lpSum([x[r][c][d] for d in D]) == 1, f"Cell_{r}_{c}_constraint"

In [31]:
# Pre-filled cells must keep their values
for r in range(9):
    for c in range(9):
        if grid_input[r][c] != 0:
            model += x[r+1][c+1][int(grid_input[r][c])] == 1, f"Prefilled_{r+1}_{c+1}_constraint"

In [32]:
# Each value must appear exactly once in each row
for r in R:
    for d in D:
        model += pulp.lpSum([x[r][c][d] for c in C]) == 1, f"Row_{r}_value_{d}_constraint"

In [33]:
# Each value must appear exactly once in each column
for c in C:
    for d in D:
        model += pulp.lpSum([x[r][c][d] for r in R]) == 1, f"Column_{c}_value_{d}_constraint"

In [34]:
# Each value must appear exactly once in each 3x3 box
for box_name, box_info in Boxes.items():
    box_rows = box_info["rows"]
    box_cols = box_info["cols"]
    for d in D:
        model += pulp.lpSum([x[r][c][d] for r in box_rows for c in box_cols]) == 1, f"Box_{box_name}_value_{d}_constraint"

#### Display / Save Formulation (Optional) 

In [15]:
#mod
#mod.writeLP("sudoku_mod.lp")

### Step 5: Run solver

In [16]:
model.solve()

1

### Step 6: Format PuLP Solution Output

In [17]:
solution_grid = np.zeros((9, 9), dtype=int)
for r in R:
    for c in C:
        for d in D:
            if x[r][c][d].varValue >= 0.99:
                solution_grid[r-1][c-1] = d
               

In [18]:
print(solution_grid)

[[3 8 1 6 2 4 9 5 7]
 [6 9 4 8 5 7 2 1 3]
 [5 2 7 3 1 9 6 8 4]
 [2 6 9 4 7 1 8 3 5]
 [1 4 8 5 6 3 7 9 2]
 [7 5 3 9 8 2 4 6 1]
 [4 1 5 7 9 8 3 2 6]
 [8 3 6 2 4 5 1 7 9]
 [9 7 2 1 3 6 5 4 8]]


---
END

# Challenging Problems

## i)
Add a difference constraint that forbids that exact solution:

If the solver is infeasible → unique solution.

If feasible → get an alternative solution → not unique.

## ii)

N-Queens (CP / IP or backtracking), and Latin squares / Magic squares can be solved with using optimization modelling

## iii) Puzzles with different difficulties

In [24]:
df_input = pd.read_excel(input_file, 
                         sheet_name="hard",
                         skiprows=0, nrows=9, 
                         usecols="A:I", header=None) #Swicth to the hard puzzle 
grid_input = df_input.fillna(0).to_numpy()

### Re-run the model and constraint code above
I only show the result here

In [35]:
model.solve()
solution_grid = np.zeros((9, 9), dtype=int)
for r in R:
    for c in C:
        for d in D:
            if x[r][c][d].varValue >= 0.99:
                solution_grid[r-1][c-1] = d
print(solution_grid)

[[2 5 8 4 9 3 1 6 7]
 [3 1 4 7 6 5 2 9 8]
 [6 9 7 1 8 2 3 5 4]
 [9 2 5 3 1 8 7 4 6]
 [4 3 6 5 7 9 8 2 1]
 [7 8 1 6 2 4 5 3 9]
 [1 6 2 9 3 7 4 8 5]
 [5 7 3 8 4 6 9 1 2]
 [8 4 9 2 5 1 6 7 3]]
